# OpenTelemetry GenAI 语义约定

OpenTelemetry 的GenAI SIG 为agent 遥测制定了标准的schema。 Span命名、属性、以及不同供应商之间上下文内容捕捉规则，使得agent 追踪在Datadog、Grafana、Jaeger、Honeycomb 等趋于同质化。

## 问题描述

每个供应商发明了自己的span命名。操作团队不得不为每种构建专属框架的仪表盘。OpenTelemetry 的GenAI SIG 通过为整个生态系统定义标准解决这个问题。

## 基本概念

### Span 分类

1. Model/Client Spans。  覆盖原生LLM调用，由供应商或者模型适配器发出。
2. Agent Spans。 `create_agent` 和 `invoke_agent`
3. Tool Spans。  每次工具调用一个，通过父子关系关联到Agent Span。

### Agent Span 命名
- Span 名称： `invoke_agent {gen_ai.agent.name}` 如果具名，否则退化到`{invoke_agent}`。
- Span 类型：
    * CLIENT ———— 用于远程agent服务（OpenAI Assitants API）
    * INTERNAL ———— 对于进程内agent框架（LangChain、CrewAI）

### 关键属性
|属性|说明||
|---|---|---|
|gen_ai.provider.name|供应商名称||
|gen_ai.request.model|请求模型ID||
|gen_ai.response.model|实际处理的模型|因为路由的原因可能与request不同|
|gen_ai.agent.name|agent标识||
|gen_ai.operation.name|操作标识|chat, completion, tool_call...|
|gen_ai.data_source.id|RAG：咨询了哪个语料或存储||

### 内容捕获

默认规则： 探针不应该按默认捕获输入和输出。通过以下option控制
- gen_ai.system_instructions
- gen_ai.input.messages
- gen_ai.output.messages

生产模式建议：外部存储内容，在span中存储引用（指针引用，而非散文）

### 什么时候失效

- Span 中包含全部提示词。  PII、机密等泄漏。需要使用外部存储
- 没有供应商名称。   多供应商的仪表盘会失败。
- Span 不包含父链接。  孤儿工具span，永远传播上下文。
- 没设稳定opt选项。  你的属性可能在后端升级时被重命名。

# 开始编码

对应本章核心：**三类 Span（model / agent / tool）**、**标准命名与 `gen_ai.*` 属性**、**默认不捕获全文（外存 + 引用）**、**父子链接防孤儿 tool span**。  
先用 OpenTelemetry SDK + 内存导出器跑通约定；再用 **LangChain + DeepSeek** 包一层真实调用（不硬凑 PyTorch）。


## 1. 教学玩具：GenAI 语义约定探针

- **命名**：`invoke_agent {name}` / `chat {model}` / `execute_tool {tool}`。
- **属性**：`gen_ai.provider.name`、`request/response.model`、`operation.name` 等。
- **内容**：默认只写 `gen_ai.input.ref` / `output.ref`；`capture_content=True` 才写 messages。
- **父子**：tool / model span 必须挂在当前 agent span 下。


In [ ]:
from __future__ import annotations

import uuid
from dataclasses import dataclass, field
from typing import Any, Callable

from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import ReadableSpan, TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.trace import SpanKind, Status, StatusCode


@dataclass
class ContentStore:
    """外存：span 只留引用，避免把全文提示词打进属性。"""

    blobs: dict[str, str] = field(default_factory=dict)

    def put(self, text: str) -> str:
        """
        Args:
            text: 完整输入/输出。

        Returns:
            ref: 内容引用 ID。
        """
        ref = f"cnt_{uuid.uuid4().hex[:10]}"
        self.blobs[ref] = text
        return ref

    def get(self, ref: str) -> str | None:
        return self.blobs.get(ref)


def agent_span_name(agent_name: str | None) -> str:
    """
    Args:
        agent_name: 具名 agent；空则退化。

    Returns:
        name: ``invoke_agent {name}`` 或 ``invoke_agent``。
    """
    if agent_name:
        return f"invoke_agent {agent_name}"
    return "invoke_agent"


def model_span_name(model: str | None) -> str:
    return f"chat {model}" if model else "chat"


def tool_span_name(tool: str | None) -> str:
    return f"execute_tool {tool}" if tool else "execute_tool"


class GenAITracer:
    """按 GenAI SIG 约定打点的薄封装。"""

    def __init__(
        self,
        *,
        service_name: str = "agent-demo",
        capture_content: bool = False,
        store: ContentStore | None = None,
    ) -> None:
        self.capture_content = capture_content
        self.store = store or ContentStore()
        self.exporter = InMemorySpanExporter()
        # 使用实例级 TracerProvider，避免全局 provider 只能 set 一次
        self.provider = TracerProvider(resource=Resource.create({"service.name": service_name}))
        self.provider.add_span_processor(SimpleSpanProcessor(self.exporter))
        self.tracer = self.provider.get_tracer("genai.semantics", "1.0.0")

    def finished(self) -> list[ReadableSpan]:
        return list(self.exporter.get_finished_spans())

    def clear(self) -> None:
        self.exporter.clear()

    def _attach_content(self, span: trace.Span, *, input_text: str | None, output_text: str | None) -> None:
        if input_text is not None:
            ref = self.store.put(input_text)
            span.set_attribute("gen_ai.input.ref", ref)
            if self.capture_content:
                span.set_attribute("gen_ai.input.messages", input_text[:2000])
        if output_text is not None:
            ref = self.store.put(output_text)
            span.set_attribute("gen_ai.output.ref", ref)
            if self.capture_content:
                span.set_attribute("gen_ai.output.messages", output_text[:2000])

    def invoke_agent(
        self,
        *,
        agent_name: str,
        provider: str,
        remote: bool = False,
        operation: str = "invoke_agent",
        input_text: str | None = None,
        fn: Callable[[], str],
    ) -> str:
        """
        Args:
            agent_name: agent 标识。
            provider: ``gen_ai.provider.name``。
            remote: True → CLIENT；进程内 → INTERNAL。
            operation: 操作名。
            input_text: 用户输入（默认只存引用）。
            fn: agent 主体。

        Returns:
            output: agent 输出。
        """
        kind = SpanKind.CLIENT if remote else SpanKind.INTERNAL
        with self.tracer.start_as_current_span(agent_span_name(agent_name), kind=kind) as span:
            span.set_attribute("gen_ai.provider.name", provider)
            span.set_attribute("gen_ai.agent.name", agent_name)
            span.set_attribute("gen_ai.operation.name", operation)
            if input_text is not None:
                self._attach_content(span, input_text=input_text, output_text=None)
            try:
                out = fn()
                self._attach_content(span, input_text=None, output_text=out)
                span.set_status(Status(StatusCode.OK))
                return out
            except Exception as e:
                span.set_status(Status(StatusCode.ERROR, str(e)))
                span.record_exception(e)
                raise

    def chat_model(
        self,
        *,
        provider: str,
        request_model: str,
        response_model: str | None = None,
        input_text: str,
        fn: Callable[[], str],
        data_source_id: str | None = None,
    ) -> str:
        """
        Model/Client span：覆盖原生 LLM 调用。

        Returns:
            text: 模型输出。
        """
        with self.tracer.start_as_current_span(model_span_name(request_model), kind=SpanKind.CLIENT) as span:
            span.set_attribute("gen_ai.provider.name", provider)
            span.set_attribute("gen_ai.request.model", request_model)
            span.set_attribute("gen_ai.operation.name", "chat")
            if data_source_id:
                span.set_attribute("gen_ai.data_source.id", data_source_id)
            self._attach_content(span, input_text=input_text, output_text=None)
            out = fn()
            span.set_attribute("gen_ai.response.model", response_model or request_model)
            self._attach_content(span, input_text=None, output_text=out)
            return out

    def execute_tool(
        self,
        *,
        tool_name: str,
        provider: str,
        args: dict[str, Any],
        fn: Callable[[], str],
    ) -> str:
        """
        Tool span：必须在 agent span 上下文中调用以建立父子关系。

        Returns:
            result: 工具结果。
        """
        with self.tracer.start_as_current_span(tool_span_name(tool_name), kind=SpanKind.INTERNAL) as span:
            span.set_attribute("gen_ai.provider.name", provider)
            span.set_attribute("gen_ai.operation.name", "tool_call")
            span.set_attribute("gen_ai.tool.name", tool_name)
            # 参数同样默认只存引用
            self._attach_content(span, input_text=str(args), output_text=None)
            out = fn()
            self._attach_content(span, input_text=None, output_text=out)
            return out


def attrs(span: ReadableSpan) -> dict[str, Any]:
    return dict(span.attributes or {})


print("GenAITracer ready | model + agent + tool spans")


## 2. 玩具示例：命名、属性、引用、父子链接


In [ ]:
def demo_genai_semantics() -> None:
    """断言 GenAI SIG 约定的关键形状。"""
    gt = GenAITracer(capture_content=False)

    secret_prompt = "SSN 123-45-6789 please summarize"

    def agent_body() -> str:
        # 进程内：先 chat，再 tool —— 都应是 agent 的子 span
        answer = gt.chat_model(
            provider="deepseek",
            request_model="deepseek-v4-flash",
            response_model="deepseek-v4-flash-routed",
            input_text=secret_prompt,
            data_source_id="kb:policies",
            fn=lambda: "need lookup",
        )
        tool_out = gt.execute_tool(
            tool_name="lookup_order",
            provider="deepseek",
            args={"order_id": "88991"},
            fn=lambda: "status=paid",
        )
        return f"{answer}; {tool_out}"

    out = gt.invoke_agent(
        agent_name="triage",
        provider="deepseek",
        remote=False,
        input_text=secret_prompt,
        fn=agent_body,
    )
    assert "paid" in out

    spans = gt.finished()
    by_name = {s.name: s for s in spans}
    assert "invoke_agent triage" in by_name
    assert "chat deepseek-v4-flash" in by_name
    assert "execute_tool lookup_order" in by_name

    agent = by_name["invoke_agent triage"]
    model = by_name["chat deepseek-v4-flash"]
    tool = by_name["execute_tool lookup_order"]

    # kind：进程内 agent = INTERNAL；model = CLIENT
    assert agent.kind == SpanKind.INTERNAL
    assert model.kind == SpanKind.CLIENT

    a = attrs(agent)
    m = attrs(model)
    t = attrs(tool)
    assert a.get("gen_ai.provider.name") == "deepseek"
    assert a.get("gen_ai.agent.name") == "triage"
    assert a.get("gen_ai.operation.name") == "invoke_agent"
    assert m.get("gen_ai.request.model") == "deepseek-v4-flash"
    assert m.get("gen_ai.response.model") == "deepseek-v4-flash-routed"
    assert m.get("gen_ai.data_source.id") == "kb:policies"
    assert t.get("gen_ai.operation.name") == "tool_call"

    # 默认不落全文；应有 ref
    assert "gen_ai.input.messages" not in a and "gen_ai.output.messages" not in a
    assert "gen_ai.input.ref" in a and "gen_ai.output.ref" in a
    assert secret_prompt not in str(dict(a))
    assert gt.store.get(a["gen_ai.input.ref"]) == secret_prompt
    print("content-by-ref ok; provider/model attrs ok")

    # 父子链接：tool/model 的 parent 是 agent
    agent_id = agent.context.span_id
    assert model.parent and model.parent.span_id == agent_id
    assert tool.parent and tool.parent.span_id == agent_id
    print("parent links ok")

    # 退化命名
    assert agent_span_name(None) == "invoke_agent"
    assert agent_span_name("billing") == "invoke_agent billing"

    # 捕获开启时才写 messages（仍建议生产关）
    gt2 = GenAITracer(capture_content=True, service_name="agent-demo-capture")
    gt2.invoke_agent(
        agent_name="x",
        provider="deepseek",
        input_text="hello",
        fn=lambda: "world",
    )
    s = gt2.finished()[0]
    assert "gen_ai.input.messages" in attrs(s)
    print("opt-in capture ok")
    print("TOY DEMO OK")


demo_genai_semantics()


## 3. 生产级：LangChain Agent + GenAI Spans + DeepSeek

用约定探针包裹 `invoke_agent` / `chat` / `execute_tool`；控制面工具：`run_traced_agent` / `export_spans` / `fetch_content`。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROVIDER = "deepseek"
PROD_GT = GenAITracer(capture_content=False, service_name="genai-prod")
LAST_OUTPUT = ""


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def lookup_order_impl(order_id: str) -> str:
    return json.dumps({"order_id": order_id, "status": "paid", "amount": 99}, ensure_ascii=False)


def run_traced_agent_impl(user_text: str, agent_name: str = "triage") -> str:
    """
    一轮：agent span 内嵌 model span + 可选 tool span。

    Returns:
        json: 输出 + 关键 span 摘要（不含全文）。
    """
    global LAST_OUTPUT, PROD_GT
    PROD_GT.clear()

    def body() -> str:
        decision = PROD_GT.chat_model(
            provider=PROVIDER,
            request_model=MODEL,
            input_text=user_text,
            fn=lambda: str(
                get_llm().invoke(
                    "You are a triage agent. If the user mentions an order id digits, "
                    'reply ONLY JSON {"action":"tool","order_id":"..."}; else '
                    '{"action":"message","content":"..."} in Chinese.\n'
                    f"User: {user_text}"
                ).content
            ).strip(),
        )
        try:
            data = json.loads(decision[decision.find("{") : decision.rfind("}") + 1])
        except Exception:
            return decision
        if data.get("action") == "tool":
            oid = str(data.get("order_id", "0"))
            tool_out = PROD_GT.execute_tool(
                tool_name="lookup_order",
                provider=PROVIDER,
                args={"order_id": oid},
                fn=lambda: lookup_order_impl(oid),
            )
            return PROD_GT.chat_model(
                provider=PROVIDER,
                request_model=MODEL,
                input_text=f"Tool:{tool_out}\nUser:{user_text}",
                fn=lambda: str(
                    get_llm()
                    .invoke(f"Summarize for user in Chinese <=40 chars.\nTool:{tool_out}")
                    .content
                ).strip(),
            )
        return str(data.get("content") or decision)

    LAST_OUTPUT = PROD_GT.invoke_agent(
        agent_name=agent_name,
        provider=PROVIDER,
        remote=False,
        input_text=user_text,
        fn=body,
    )
    summary = []
    for s in PROD_GT.finished():
        summary.append(
            {
                "name": s.name,
                "kind": str(s.kind),
                "parent": None if s.parent is None else format(s.parent.span_id, "016x"),
                "span_id": format(s.context.span_id, "016x"),
                "attrs": {
                    k: v
                    for k, v in attrs(s).items()
                    if k.startswith("gen_ai.") and not k.endswith(".messages")
                },
            }
        )
    return json.dumps({"output": LAST_OUTPUT, "spans": summary}, ensure_ascii=False)


def export_spans_impl() -> str:
    """
    Returns:
        json: 已完成 span 列表（无 messages 全文）。
    """
    rows = []
    for s in PROD_GT.finished():
        a = attrs(s)
        rows.append(
            {
                "name": s.name,
                "provider": a.get("gen_ai.provider.name"),
                "operation": a.get("gen_ai.operation.name"),
                "has_input_messages": "gen_ai.input.messages" in a,
                "input_ref": a.get("gen_ai.input.ref"),
                "output_ref": a.get("gen_ai.output.ref"),
                "parent": None if s.parent is None else format(s.parent.span_id, "016x"),
            }
        )
    return json.dumps({"spans": rows}, ensure_ascii=False)


def fetch_content_impl(ref: str) -> str:
    """
    按引用取外存内容（演示「span 存指针」）。

    Returns:
        json: 内容或 missing。
    """
    text = PROD_GT.store.get(ref)
    if text is None:
        return json.dumps({"error": "missing", "ref": ref})
    return json.dumps({"ref": ref, "text": text}, ensure_ascii=False)


class RunArgs(BaseModel):
    text: str
    agent_name: str = "triage"


class FetchArgs(BaseModel):
    ref: str = Field(description="gen_ai.input.ref or output.ref")


class EmptyArgs(BaseModel):
    pass


def build_otel_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: 追踪控制面。
    """

    def _run(**kwargs: Any) -> str:
        a = RunArgs(**kwargs)
        return run_traced_agent_impl(a.text, a.agent_name)

    def _export(**kwargs: Any) -> str:
        return export_spans_impl()

    def _fetch(**kwargs: Any) -> str:
        return fetch_content_impl(FetchArgs(**kwargs).ref)

    return [
        StructuredTool.from_function(
            name="run_traced_agent",
            description="Run agent with GenAI semantic spans (model/agent/tool).",
            func=_run,
            args_schema=RunArgs,
        ),
        StructuredTool.from_function(
            name="export_spans",
            description="Export finished spans without full prompt text.",
            func=_export,
            args_schema=EmptyArgs,
        ),
        StructuredTool.from_function(
            name="fetch_content",
            description="Fetch full content from external store by ref.",
            func=_fetch,
            args_schema=FetchArgs,
        ),
    ]


OTEL_TOOLS = build_otel_tools()


def build_control_agent() -> Any:
    """
    Returns:
        agent: 通过工具驱动带 GenAI 约定的运行。
    """
    system = (
        "You operate a GenAI-instrumented agent runtime.\n"
        "Flow: run_traced_agent -> export_spans -> optionally fetch_content(ref).\n"
        "Explain span types and that prompts are refs-only by default. Chinese."
    )
    return create_agent(get_llm(), OTEL_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 600 else str(m.content)[:600] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


print(f"GenAI OTel + LangChain ready | {MODEL}")


## 4. 生产示例：追踪一轮 Agent 并导出 Span


In [ ]:
def demo_deepseek_genai_otel() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    print("=== traced run ===")
    raw = run_traced_agent_impl("帮我查订单 88991 的状态", agent_name="triage")
    data = json.loads(raw)
    print(json.dumps(data, ensure_ascii=False, indent=2)[:1000])
    assert data.get("output")
    names = [s["name"] for s in data["spans"]]
    assert any(n.startswith("invoke_agent") for n in names)
    assert any(n.startswith("chat") for n in names)
    # tool span 视模型是否选择查单
    exported = json.loads(export_spans_impl())
    assert all(not s.get("has_input_messages") for s in exported["spans"])
    assert all(s.get("provider") == "deepseek" for s in exported["spans"] if s.get("provider"))
    print("export:", json.dumps(exported, ensure_ascii=False)[:600])

    # 用引用取回原文（外存）
    ref = next(s["input_ref"] for s in exported["spans"] if s.get("input_ref"))
    fetched = json.loads(fetch_content_impl(ref))
    assert "88991" in fetched.get("text", "")
    print("fetch_content ok")

    print("=== control agent ===")
    try:
        agent = build_control_agent()
        result = agent.invoke(
            {
                "messages": [
                    HumanMessage(
                        content=(
                            "run_traced_agent 问一个带订单号的问题，再 export_spans，"
                            "用中文说明有哪些 span、是否包含全文 messages。"
                        )
                    )
                ]
            }
        )
        print(format_agent_messages(result["messages"]))
        assert count_tool_calls(result["messages"]) >= 2
    except Exception as e:
        print(f"control agent skipped due to LLM error: {type(e).__name__}: {e}")
    print("PRODUCTION DEMO OK")


demo_deepseek_genai_otel()
